In [1]:
import pandas as pd
import os
from functools import reduce
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
root_url = 'data/medical/2025'
sheet_filepaths = sorted(
    [
        os.path.join(root_url, f) 
        for f in os.listdir(root_url) 
        if f.endswith('.xlsx')
    ]
)

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

In [3]:
def create_column_names(original_column_names):
    """Add the _M, _F, _Total, _Percent to all indicators"""
    code_column_names = ['PSGC', 'Area']

    for indicator in original_column_names:
        code_column_names.append(indicator + '_M')
        code_column_names.append(indicator + '_F')
        code_column_names.append(indicator + '_Total')
        code_column_names.append(indicator + '_Percent')

    return code_column_names

In [4]:
COLUMN_ORDER = (
    create_column_names(
        [
            'BCG', 'HEPA_B1', 'CPAB', 
            'DPT_1', 'DPT_2', 'DPT_3',
            'OPV_1', 'OPV_2', 'OPV_3',
            'IPV_1', 'IPV_2',
            'PCV_1', 'PCV_2', 'PCV_3',
            'FIC', 'CIC'
        ]
    )
)

COLUMN_ORDER.insert(2, 'Month')

GEO_COLUMN_ORDER = COLUMN_ORDER.copy()
GEO_COLUMN_ORDER.insert(3, 'Level')
GEO_COLUMN_ORDER.insert(4, 'Income')

In [5]:
def extract_monthly_data(
    sheet_filepath,
    cols_to_drop,
    col_indicators,
    months=None,
    is_verbose=False,
):
    """
    Extract and clean monthly data from an Excel file, then combine into one DataFrame.
    """

    # Default to all months if none are provided
    if months is None:
        months = [
            "Jan", "Feb", "Mar", "Apr", "May", "Jun",
            "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
        ]

    table_month_dfs = []

    for month in months:
        if is_verbose:
            print(month)

        # Read the specific month's sheet
        table_month_df = pd.read_excel(
            sheet_filepath,
            sheet_name=month,
            skiprows=6,
            skipfooter=4,
        )

        # Drop unwanted columns by index
        table_month_df.drop(
            columns=table_month_df.columns[cols_to_drop],
            inplace=True,
        )

        # Rename columns based on indicator structure
        table_month_df.columns = create_column_names(col_indicators)

        # Add month identifier
        table_month_df["Month"] = month

        # Store processed DataFrame
        table_month_dfs.append(table_month_df)

    # Combine all months into a single DataFrame
    return pd.concat(table_month_dfs, ignore_index=True)

In [6]:
table_configs = {
    "table1": {
        "sheet_filepath": sheet_filepaths[0],
        "cols_to_drop": [2] + list(range(7, 11)) + list(range(15, 20)),
        "col_indicators": ['BCG', 'HEPA_B1', 'CPAB'],
    },
    "table2": {
        "sheet_filepath": sheet_filepaths[1],
        "cols_to_drop": [2] + list(range(15, 28)),
        "col_indicators": ['DPT_1', 'DPT_2', 'DPT_3'],
    },
    "table3": {
        "sheet_filepath": sheet_filepaths[2],
        "cols_to_drop": [2] + list(range(15, 28)),
        "col_indicators": ['OPV_1', 'OPV_2', 'OPV_3'],
    },
    "table4": {
        "sheet_filepath": sheet_filepaths[3],
        "cols_to_drop": [2] + list(range(11, 20)),
        "col_indicators": ['IPV_1', 'IPV_2'],
    },
    "table5": {
        "sheet_filepath": sheet_filepaths[4],
        "cols_to_drop": [2] + list(range(15, 28)),
        "col_indicators": ['PCV_1', 'PCV_2', 'PCV_3'],
    },
    "table6": {
        "sheet_filepath": sheet_filepaths[5],
        "cols_to_drop": [2] + list(range(11, 20)),
        "col_indicators": ['MCV_1', 'MCV_2'],
    },
    "table6": {
        "sheet_filepath": sheet_filepaths[6],
        "cols_to_drop": [2, 7],
        "col_indicators": ['FIC', 'CIC'],
    },
}

In [7]:
table_dfs = []
for table, config in table_configs.items():
    print(table)
    table_df = extract_monthly_data(**config)
    table_dfs.append(table_df)

table1
table2
table3
table4
table5
table6


In [8]:
# there are misspellings right now
corrections = {
    "PagsanJun": "Pagsanjan",
    "Juniuay": "Janiuay",
    "PinamungaJun": "Pinamungajan",
    "PambuJun": "Pambujan",
    "NauJun": "Naujan",
    "Maguindanao Sur": "Maguindanao del Sur",
}

for table_df in table_dfs:
    table_df.Area = table_df.Area.replace(corrections)

In [9]:
# merge the table dataframes by PSGC, Area, and Month
monthly_df = reduce(
    lambda left, right: pd.merge(
        left, 
        right, 
        on=["PSGC", "Area", "Month"], 
        how="outer",
    ),
    table_dfs,
)

# Impute all * into the median
for col in monthly_df.columns:
    if col not in ["PSGC", "Area", "Month"]:
        # Replace "*" safely without dtype guessing
        monthly_df[col] = monthly_df[col].mask(
            monthly_df[col] == "*", np.nan
        )

        # Explicit conversion
        monthly_df[col] = pd.to_numeric(monthly_df[col], errors="coerce")

        # Fill with median
        monthly_df[col] = monthly_df[col].fillna(monthly_df[col].median())

# set an order for the months
month_order = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]
monthly_df["Month"] = pd.Categorical(
    monthly_df["Month"],
    categories=month_order,
    ordered=True
)

# make sure the PSGC is a string and has 10 digits
monthly_df.PSGC = monthly_df.PSGC.astype(str).str.zfill(10)
monthly_df = monthly_df[COLUMN_ORDER].sort_values(by=['PSGC', 'Month'], ascending=[True, True]).copy()

In [10]:
psgc_df = pd.read_excel('data/income/psgc-1q-2025-publication-datafile.xlsx', sheet_name='PSGC')
psgc_df = psgc_df[['10-digit PSGC', 'Geographic Level', 'Income\nClassification']]
psgc_df.columns = ['PSGC', 'Level', 'Income']
psgc_df['PSGC'] = psgc_df['PSGC'].astype(str).str.zfill(10)

psgc_df = psgc_df[
    (psgc_df.Level == 'Mun') |
    (psgc_df.Level== 'City') | 
    (psgc_df.Level == 'Prov')
].copy()

In [11]:
# update zamboanga_sibugay and alburqueqe's psgc code
psgc_fixes = {
    "Zamboanga Sibugay": "0908300000",
    "Alburquerque": "0701201000",
}

for area, psgc in psgc_fixes.items():
    monthly_df.loc[monthly_df["Area"] == area, "PSGC"] = psgc
    
monthly_psgc_df = monthly_df.merge(psgc_df, on='PSGC', how='left')
monthly_psgc_df = monthly_psgc_df[GEO_COLUMN_ORDER].copy()

The rows that have its income missing are for regions and the special geographic area. 

In [12]:
is_income_null = monthly_psgc_df.Income.isnull()
is_region = monthly_psgc_df.PSGC.str.endswith('00000000')
monthly_psgc_df[is_income_null & ~is_region]

,PSGC,Area,Month,Level,Income,BCG_M,BCG_F,BCG_Total,BCG_Percent,HEPA_B1_M,HEPA_B1_F,HEPA_B1_Total,HEPA_B1_Percent,CPAB_M,CPAB_F,CPAB_Total,CPAB_Percent,DPT_1_M,DPT_1_F,DPT_1_Total,DPT_1_Percent,DPT_2_M,DPT_2_F,DPT_2_Total,DPT_2_Percent,DPT_3_M,DPT_3_F,DPT_3_Total,DPT_3_Percent,OPV_1_M,OPV_1_F,OPV_1_Total,OPV_1_Percent,OPV_2_M,OPV_2_F,OPV_2_Total,OPV_2_Percent,OPV_3_M,OPV_3_F,OPV_3_Total,OPV_3_Percent,IPV_1_M,IPV_1_F,IPV_1_Total,IPV_1_Percent,IPV_2_M,IPV_2_F,IPV_2_Total,IPV_2_Percent,PCV_1_M,PCV_1_F,PCV_1_Total,PCV_1_Percent,PCV_2_M,PCV_2_F,PCV_2_Total,PCV_2_Percent,PCV_3_M,PCV_3_F,PCV_3_Total,PCV_3_Percent,FIC_M,FIC_F,FIC_Total,FIC_Percent,CIC_M,CIC_F,CIC_Total,CIC_Percent
20796,1999900000,Special Geographic Area (SGA),Jan,NaN,NaN,36.0,27.0,63,0.915432,40.0,47.0,87,1.264167,63.0,57.0,120,1.905064,156.0,155,311,4.519035,117,137,254,3.690788,114,122,236,3.429236,168,156,324,4.707934,110,135,245,3.560012,106,115,221,3.211276,101,133.0,234,3.400174,81.0,91,172,2.499273,40,44,84,1.220575,28,25,53,0.770125,36,26,62,0.900901,128.0,113.0,241,3.498331,12,16,28,0.595618
20797,1999900000,Special Geographic Area (SGA),Feb,NaN,NaN,289.0,304.0,593,8.616681,50.0,54.0,104,1.511189,75.0,97.0,172,2.730592,200.0,200,400,5.812264,127,149,276,4.010462,146,150,296,4.301075,224,217,441,6.408021,165,177,342,4.969486,165,176,341,4.954955,126,102.0,228,3.312990,111.0,87,198,2.877071,353,389,742,10.781749,108,137,245,3.560012,104,101,205,2.978785,135.0,138.0,273,3.962839,14,10,24,0.510530
20798,1999900000,Special Geographic Area (SGA),Mar,NaN,NaN,222.0,235.0,457,6.640511,66.0,89.0,155,2.252252,115.0,131.0,246,3.905382,225.0,247,472,6.858471,217,183,400,5.812264,166,189,355,5.158384,227,230,457,6.640511,236,215,451,6.553328,181,212,393,5.710549,107,122.0,229,3.327521,129.0,111,240,3.487358,305,329,634,9.212438,250,236,486,7.061901,103,126,229,3.327521,147.0,142.0,289,4.195094,24,21,45,0.957243
20799,1999900000,Special Geographic Area (SGA),Apr,NaN,NaN,93.0,115.0,208,3.022377,24.0,28.0,52,0.755594,61.0,74.0,135,2.143197,165.0,179,344,4.998547,214,199,413,6.001162,200,178,378,5.492589,202,202,404,5.870387,227,232,459,6.669573,247,215,462,6.713165,124,122.0,246,3.574542,93.0,79,172,2.499273,188,220,408,5.928509,246,231,477,6.931125,188,145,333,4.838710,155.0,124.0,279,4.049935,31,28,59,1.255052
20800,1999900000,Special Geographic Area (SGA),May,NaN,NaN,66.0,73.0,139,2.019762,22.0,14.0,36,0.523104,67.0,58.0,125,1.984442,154.0,140,294,4.272014,131,151,282,4.097646,157,153,310,4.504505,165,156,321,4.664342,138,152,290,4.213891,174,192,366,5.318221,115,135.0,250,3.632665,74.0,93,167,2.426620,138,134,272,3.952339,151,192,343,4.984016,153,189,342,4.969486,139.0,146.0,285,4.137030,15,10,25,0.531802
20801,1999900000,Special Geographic Area (SGA),Jun,NaN,NaN,62.0,79.0,141,2.048823,47.0,36.0,83,1.206045,112.0,112.0,224,3.556120,181.0,172,353,5.129323,161,174,335,4.867771,175,189,364,5.289160,186,195,381,5.536181,164,185,349,5.071200,181,206,387,5.623365,155,145.0,300,4.359198,81.0,98,179,2.600988,204,173,377,5.478059,199,191,390,5.666957,194,228,422,6.131938,177.0,196.0,373,5.414429,16,14,30,0.638162
20802,1999900000,Special Geographic Area (SGA),Jul,NaN,NaN,90.0,89.0,179,2.600988,64.0,57.0,121,1.758210,106.0,124.0,230,3.651373,180.0,202,382,5.550712,188,173,361,5.245568,140,162,302,4.388259,185,188,373,5.419936,201,192,393,5.710549,142,162,304,4.417321,126,135.0,261,3.792502,82.0,124,206,2.993316,189,192,381,5.536181,175,156,331,4.809648,152,155,307,4.460913,159.0,164.0,323,4.688634,17,15,32,0.680706
20803,1999900000,Special Geographic Area (SGA),Aug,NaN,NaN,132.0,109.0,241,3.501889,72.0,71.0,143,2.077884,138.0,139.0,277,4.397523,193.0,189,382,5.550712,178,188,366,5.318221,185,201,386,5.608835,191,174,365,5.303691,178,194,372,5.405405,203,208,411,5.972101,118,131.0,249,3.618134,117.0,114,231,3.356582,168,154,322,4.678872,181,172,353,5.129323,150,163,313,4.548096,168.0,165.0,333,4.833793,12,16,28,0.595618
20804,1999900000,Special Geographic Area (SGA),Sep,NaN,NaN,93.

# Output Time

In [13]:
# Columns that should stay single (not grouped)
base_cols = ['PSGC', 'Area', 'Month', 'Level', 'Income']


def write_with_grouped_headers(df, writer, sheet_name):
    df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1)
    
    workbook  = writer.book
    worksheet = writer.sheets[sheet_name]
    
    # Formats
    header_main = workbook.add_format({
        'bold': True,
        'align': 'center',
        'valign': 'middle',
        'border': 1,
        'bg_color': '#1D4E79',
        'font_color': '#FFFFFF'
    })

    header_sub = workbook.add_format({
        'bold': True,
        'align': 'center',
        'border': 1,
        'bg_color': '#1D4E79',
        'font_color': '#FFFFFF'
    })
        
    highlight_format = workbook.add_format({
        'bg_color': '#FFF2CC'  # light yellow
    })
    
    # ---- STEP 1: Handle base columns ----
    col_idx = 0
    for col in df.columns:
        if col in base_cols:
            worksheet.merge_range(0, col_idx, 1, col_idx, col, header_main)
            col_idx += 1
    
    # ---- STEP 2: Dynamically group remaining columns ----
    grouped_cols = [c for c in df.columns if c not in base_cols]
    
    # Extract prefix (before last "_")
    from collections import defaultdict
    groups = defaultdict(list)
    
    for col in grouped_cols:
        if '_' in col:
            prefix = '_'.join(col.split('_')[:-1])  # e.g. OPV_M → OPV
            groups[prefix].append(col)
        else:
            groups[col].append(col)
    
    # Write grouped headers
    for group, cols in groups.items():
        start_col = col_idx
        end_col = col_idx + len(cols) - 1
        
        # Top header
        worksheet.merge_range(0, start_col, 0, end_col, group, header_main)
        
        # Subheaders
        for i, col in enumerate(cols):
            sub = col.split('_')[-1] if '_' in col else col
            worksheet.write(1, start_col + i, sub, header_sub)
        
        col_idx += len(cols)
    
    # ---- STEP 3: Highlight rows where PSGC ends with 00000000 ----
    if 'PSGC' in df.columns:
        psgc_col_idx = df.columns.get_loc('PSGC')
        
        n_rows = len(df)
        n_cols = len(df.columns)
        
        # Helper to convert column index → Excel letter (handles AA, AB, etc.)
        def colnum_to_excel(n):
            string = ""
            while n >= 0:
                string = chr(n % 26 + 65) + string
                n = n // 26 - 1
            return string
        
        col_letter = colnum_to_excel(psgc_col_idx)
        
        highlight_format = workbook.add_format({
            'bg_color': '#A6A6A6',
            'bold': True
        })
    
    worksheet.conditional_format(
        2, 0,                     # start row, start col
        n_rows + 1, n_cols - 1,   # end row, end col
        {
            'type': 'formula',
            'criteria': f'=RIGHT(TEXT(${col_letter}3,"0"),8)="00000000"',
            'format': highlight_format
        }
    )
    
    # ---- STEP 4: Formatting ----
    worksheet.freeze_panes(2, 0)
    
    for i, col in enumerate(df.columns):
        worksheet.set_column(i, i, 18)


# ---- WRITE FILE ----
with pd.ExcelWriter('outputs/monthly_immunization.xlsx', engine='xlsxwriter') as writer:
    
    # All data
    write_with_grouped_headers(monthly_psgc_df, writer, 'All_Data')
    
    # Per year
    months = monthly_psgc_df['Month'].unique()
    
    for month in months:
        monthly_df = monthly_psgc_df[
            monthly_psgc_df['Month'] == month
        ]
        
        write_with_grouped_headers(monthly_df, writer, str(month))

In [14]:
monthly_psgc_df.to_csv('outputs/monthly_immunization.csv', index=False)